# 15. Dealing with Imbalanced Classes

## 15.1 The Imbalanced Data Problem

When one class significantly outnumbers the other, standard classifiers become biased toward the majority class. **Accuracy** becomes a misleading metric — a model that predicts only the majority class can achieve 95% accuracy on a 95:5 dataset while completely failing to detect the minority class.

Below we simulate an imbalanced dataset and demonstrate why accuracy alone is insufficient.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, weights=[0.95],
                           flip_y=0, random_state=42)
print ('Feature matrix shape: %s\n'%str(X.shape))
print ('Target vector shape: %s\n'%str(y.shape))
print ('Class distribution:\n%s\n'%pd.Series(y).value_counts().to_string())

Feature matrix shape: (1000, 2)
Target vector shape: (1000,)
Class distribution:
0    950
1     50


In [2]:
from sklearn.dummy import DummyClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

acc_dummy = accuracy_score(y_test, y_pred_dummy)
rec_dummy = recall_score(y_test, y_pred_dummy)
print ('Dummy Classifier (always predicts majority class):\n')
print ('Accuracy: %.4f\n'%acc_dummy)
print ('Recall for class 1: %.4f\n'%rec_dummy)

Dummy Classifier (always predicts majority class):
Accuracy: 0.9500
Recall for class 1: 0.0000


<hr>

## 15.2 Class Weight Adjustment

Many `sklearn` estimators accept a `class_weight` parameter. Setting `class_weight='balanced'` automatically adjusts weights inversely proportional to class frequencies, forcing the model to pay more attention to the minority class.

Below we compare `sklearn.**LogisticRegression**(*class_weight=None*)` with `sklearn.**LogisticRegression**(*class_weight='balanced'*)`.

In [3]:
lr_default = LogisticRegression(class_weight=None, random_state=42)
lr_default.fit(X_train, y_train)
y_pred_default = lr_default.predict(X_test)

print ('Default LogisticRegression (class_weight=None):\n')
print ('Accuracy: %.4f\n'%accuracy_score(y_test, y_pred_default))
print ('Classification Report:\n%s\n'%classification_report(y_test, y_pred_default))

Default LogisticRegression (class_weight=None):
Accuracy: 0.9500

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       190
           1       0.00      0.00      0.00        10

    accuracy                           0.95       200
   macro avg       0.48      0.50      0.49       200
weighted avg       0.90      0.95      0.93       200



In [4]:
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42)
lr_balanced.fit(X_train, y_train)
y_pred_balanced = lr_balanced.predict(X_test)

print ('Balanced LogisticRegression (class_weight=\'balanced\'):\n')
print ('Accuracy: %.4f\n'%accuracy_score(y_test, y_pred_balanced))
print ('Classification Report:\n%s\n'%classification_report(y_test, y_pred_balanced))

Balanced LogisticRegression (class_weight='balanced'):
Accuracy: 0.8750

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.88      0.93       190
           1       0.28      0.70      0.40        10

    accuracy                           0.88       200
   macro avg       0.63      0.79      0.66       200
weighted avg       0.94      0.88      0.90       200



<hr>

## 15.3 Random Forest with Balanced Weights

Tree-based models also support `class_weight`. Using `sklearn.**RandomForestClassifier**(*class_weight='balanced'*)` penalizes misclassifications of the minority class more heavily during tree construction.

Below we compare a default random forest with a balanced one.

In [5]:
rf_default = RandomForestClassifier(class_weight=None, random_state=42)
rf_default.fit(X_train, y_train)
y_pred_rf_default = rf_default.predict(X_test)

print ('Default Random Forest (class_weight=None):\n')
print ('Accuracy: %.4f\n'%accuracy_score(y_test, y_pred_rf_default))
print ('Classification Report:\n%s\n'%classification_report(y_test, y_pred_rf_default))

Default Random Forest (class_weight=None):
Accuracy: 0.9450

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       190
           1       0.00      0.00      0.00        10

    accuracy                           0.95       200
   macro avg       0.48      0.50      0.49       200
weighted avg       0.90      0.95      0.92       200



In [6]:
rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_balanced.fit(X_train, y_train)
y_pred_rf_balanced = rf_balanced.predict(X_test)

print ('Balanced Random Forest (class_weight=\'balanced\'):\n')
print ('Accuracy: %.4f\n'%accuracy_score(y_test, y_pred_rf_balanced))
print ('Classification Report:\n%s\n'%classification_report(y_test, y_pred_rf_balanced))

Balanced Random Forest (class_weight='balanced'):
Accuracy: 0.9150

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.94      0.95       190
           1       0.33      0.60      0.43        10

    accuracy                           0.92       200
   macro avg       0.65      0.77      0.69       200
weighted avg       0.94      0.92      0.93       200



<hr>

## 15.4 SMOTE (Synthetic Minority Over-sampling)

**SMOTE** creates synthetic samples for the minority class by interpolating between existing minority instances and their nearest neighbours, rather than simply duplicating data.

Below we generate a fresh imbalanced dataset, inspect the class distribution, apply SMOTE, and compare the result.

In [7]:
X_sm, y_sm = make_classification(n_samples=1000, n_features=2, n_redundant=0,
                                 n_clusters_per_class=1, weights=[0.95],
                                 flip_y=0, random_state=42)

print ('Imbalanced dataset for SMOTE:\n')
print ('Feature matrix shape: %s\n'%str(X_sm.shape))
print ('Before SMOTE:\n%s\n'%pd.Series(y_sm).value_counts().to_string())

Imbalanced dataset for SMOTE:
Feature matrix shape: (1000, 2)

Before SMOTE:
0    950
1     50



In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_sm, y_sm)

print ('After SMOTE:\n%s\n'%pd.Series(y_res).value_counts().to_string())
print ('Feature matrix shape after SMOTE: %s\n'%str(X_res.shape))

After SMOTE:
0    950
1    950

Feature matrix shape after SMOTE: (1900, 2)



<hr>

## 15.5 Evaluation Metrics for Imbalanced Data

When classes are imbalanced, we rely on **Precision**, **Recall**, **F1-Score**, and **ROC-AUC** rather than accuracy. The **confusion matrix** also provides a detailed breakdown of correct and incorrect predictions per class.

In [9]:
precision = precision_score(y_test, y_pred_rf_balanced)
recall = recall_score(y_test, y_pred_rf_balanced)
f1 = f1_score(y_test, y_pred_rf_balanced)
roc_auc = roc_auc_score(y_test, rf_balanced.predict_proba(X_test)[:, 1])

print ('Balanced Random Forest evaluation:\n')
print ('Precision: %.4f\n'%precision)
print ('Recall: %.4f\n'%recall)
print ('F1 Score: %.4f\n'%f1)
print ('ROC-AUC: %.4f\n'%roc_auc)

Balanced Random Forest evaluation:

Precision: 0.3333
Recall: 0.6000
F1 Score: 0.4286
ROC-AUC: 0.8653



In [10]:
cm = confusion_matrix(y_test, y_pred_rf_balanced)
print ('Confusion Matrix:\n%s\n'%str(cm))

Confusion Matrix:
[[179  11]
 [  4   6]]



<hr>

## 15.6 Threshold Tuning

By default, `sklearn` classifiers use a **0.5 decision threshold** for binary classification. Lowering this threshold makes the model more sensitive to the minority class (higher recall), but may increase false positives (lower precision).

Below we vary the threshold and observe the trade-off.

In [11]:
y_probs = rf_balanced.predict_proba(X_test)[:, 1]
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print ('Threshold tuning for Balanced Random Forest:\n')
for thresh in thresholds:
    y_pred_thresh = (y_probs >= thresh).astype(int)
    p = precision_score(y_test, y_pred_thresh)
    r = recall_score(y_test, y_pred_thresh)
    f = f1_score(y_test, y_pred_thresh)
    print ('Threshold: %.2f - Precision: %.4f, Recall: %.4f, F1: %.4f\n'%(thresh, p, r, f))

Threshold tuning for Balanced Random Forest:

Threshold: 0.30 - Precision: 0.2857, Recall: 0.8000, F1: 0.4211
Threshold: 0.40 - Precision: 0.3333, Recall: 0.6000, F1: 0.4286
Threshold: 0.50 - Precision: 0.3333, Recall: 0.6000, F1: 0.4286
Threshold: 0.60 - Precision: 0.5000, Recall: 0.5000, F1: 0.5000
Threshold: 0.70 - Precision: 0.5000, Recall: 0.4000, F1: 0.4444
Threshold: 0.80 - Precision: 0.5000, Recall: 0.2000, F1: 0.2857



<hr>

## 15.7 Assignment

1. Generate a dataset with a **99:1** class imbalance using `make_classification`.
2. Split into train/test sets (80:20, stratify).
3. Train a `**RandomForestClassifier**(*class_weight='balanced_subsample'*)` and a `**LogisticRegression**(*class_weight='balanced'*)`.
4. Evaluate both models using **Precision**, **Recall**, **F1**, and **ROC-AUC**.
5. Apply **SMOTE** and re-train the same models on the resampled data.
6. Compare the performance before and after SMOTE in a short summary.

Complete the code template below:

In [12]:
# ---------- Assignment Solution ----------

# 1. Generate dataset with 99:1 imbalance
# X_ass, y_ass = make_classification(...)

# 2. Train/test split
# X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(...)

# 3. Models
# rf_ass = RandomForestClassifier(class_weight='balanced_subsample', random_state=42)
# lr_ass = LogisticRegression(class_weight='balanced', random_state=42)

# 4. Evaluate
# print metrics

# 5. SMOTE
# smote_ass = SMOTE(random_state=42)
# X_res_a, y_res_a = smote_ass.fit_resample(X_train_a, y_train_a)

# 6. Re-train on resampled data and compare
# print comparison summary